# Day 2: LLM API Parameters & Prompt Designs
In this notebook, we will learn how to control our local LLM's creativity and request structured outputs.

## What You'll Learn:
1. **Temperature:** Controlling how creative or deterministic (predictable) the AI is.
2. **Structured Formats:** Asking the LLM to format its answers as JSON or Markdown tables.
3. **Prompt Templates:** Using simple Python strings to build reusable prompt structures.

In [2]:
# Import the ollama library to communicate with local models
import ollama

print("Day 2 workspace loaded successfully!")

Day 2 workspace loaded successfully!


## What is Temperature?
**Temperature** is a parameter that controls how creative or random the model's responses are. It usually ranges from `0.0` to `1.0`.

- **Temperature = 0.0 (Deterministic / Predictable):** The model will always choose the most likely words. Use this for tasks that require precision, like code writing, math, or factual lookup.
- **Temperature = 1.0 (Creative / Random):** The model will take more risks and choose more unusual words. Use this for storytelling, brainstorming, or roleplaying.

Let's test this in Python! We will ask the model to complete the same sentence starter twice: once with low temperature, and once with high temperature.

In [3]:
# The starter sentence we want the model to complete
prompt = "Complete this sentence in a creative way: 'The robot walked into the kitchen and...'"

# ------------------------------------------------------------
# 1. Low Temperature (0.0) -> Highly predictable
# ------------------------------------------------------------
print("Low Temperature (0.0) - Predictable & Safe:")
print("-" * 50)

response_low = ollama.chat(
    model='llama3.2:1b',
    messages=[{'role': 'user', 'content': prompt}],
    options={'temperature': 0.0}   # We set temperature here!
)
print(response_low.message.content)


print("\n" + "="*60 + "\n")


# ------------------------------------------------------------
# 2. High Temperature (1.0) -> Creative & Random
# ------------------------------------------------------------
print("High Temperature (1.0) - Creative & Random:")
print("-" * 50)

response_high = ollama.chat(
    model='llama3.2:1b',
    messages=[{'role': 'user', 'content': prompt}],
    options={'temperature': 1.0}   # We set temperature here!
)
print(response_high.message.content)

Low Temperature (0.0) - Predictable & Safe:
--------------------------------------------------
...suddenly found itself face to face with a mysterious, steaming bowl of "Mother's Special Sauce," which emitted a soft hum that harmonized with the whirring of its mechanical limbs, causing it to pause mid-step and ponder the meaning behind this culinary enigma.


High Temperature (1.0) - Creative & Random:
--------------------------------------------------
...suddenly materialized from the walls, its shiny metallic limbs unfolding like a puzzle piece as it whirred to life, the fluorescent lights above casting an eerie glow on its glowing blue eyes as it fixed its gaze on the sizzling bacon in the skillet.


## Requesting Structured Formats (JSON)
Often, we need the LLM to extract data and return it in a format our code can read. 

**JSON (JavaScript Object Notation)** is the industry-standard format for data exchange. It represents data as key-value pairs (very similar to Python dictionaries).

To get clean JSON from an LLM:
1. We ask the model to format its output as JSON in our prompt instructions.
2. In Ollama, we can also pass `'format': 'json'` in the API call options to force the model to only output valid JSON.

In [4]:
import json

# Standard text we want to extract information from
raw_text = "Apple Inc. was founded by Steve Jobs in 1976 in Cupertino, California."

# Simple prompt asking the model to extract and return ONLY JSON
prompt = f"""
Extract the company name, founder, and year founded from the text below.
Return the result as a JSON object with these keys: company, founder, year.

Text: "{raw_text}"
"""

print("Requesting JSON extraction...")

# Call the model, forcing it to output valid JSON format
response = ollama.chat(
    model='llama3.2:1b',
    messages=[{'role': 'user', 'content': prompt}],
    format='json'  # This option tells Ollama to guarantee a valid JSON response!
)

raw_reply = response.message.content
print("\Raw Response from AI:")
print(raw_reply)

# ------------------------------------------------------------
# Parse the JSON in Python so we can use it!
# ------------------------------------------------------------
print("\Loading the response in Python:")
try:
    # Convert the string response into a Python dictionary
    data = json.loads(raw_reply)
    
    # Access keys inside the dictionary just like regular Python code
    print(f"Company Name: {data['company']}")
    print(f"Founder:      {data['founder']}")
    print(f"Year:         {data['year']}")
    
except json.JSONDecodeError as e:
    print(f"Failed to parse JSON: {e}")

Requesting JSON extraction...
\Raw Response from AI:
{
    "company": "Apple Inc.",
    "founder": "Steve Jobs",
    "year": 1976
}
\Loading the response in Python:
Company Name: Apple Inc.
Founder:      Steve Jobs
Year:         1976


## Reusable Prompt Templates
Instead of copy-pasting the same instructions every time, we define a blueprint template string. 

We use Python placeholders like `{topic}` or `{style}` inside our prompt. Then, we use `.format()` to plug in different values dynamically. 

This makes our code highly reusable and clean.

In [5]:
# 1. Define a template prompt with placeholders {language} and {phrase}
translation_template = """
Translate the following phrase into {language}. 
Provide ONLY the translated words and nothing else.

Phrase: "{phrase}"
"""

# 2. Let's write a simple helper function that fills the template and calls Ollama
def translate_phrase(target_language, input_phrase):
    # Fill in the placeholders in our template
    filled_prompt = translation_template.format(
        language=target_language,
        phrase=input_phrase
    )
    
    # Send the filled prompt to Ollama
    response = ollama.chat(
        model='llama3.2:1b',
        messages=[{'role': 'user', 'content': filled_prompt}]
    )
    
    # Return the translation text
    return response.message.content.strip()

# ------------------------------------------------------------
# 3. Test our template with different inputs!
# ------------------------------------------------------------
phrase_to_translate = "Hello, how are you today?"

# Test Hindi
hindi_result = translate_phrase(target_language="Hindi", input_phrase=phrase_to_translate)
print(f"\n🇪🇸 Hindi translation: {hindi_result}")

# Test Gujarati
gujarati_result = translate_phrase(target_language="Gujarati", input_phrase=phrase_to_translate)
print(f"\n🇫🇷 Gujarati translation: {gujarati_result}")

# Test Panjabi
panjabi_result = translate_phrase(target_language="Panjabi", input_phrase=phrase_to_translate)
print(f"\n🇯🇵 Panjabi translation: {panjabi_result}")


🇪🇸 Hindi translation: मैं माफ करूंगा
ना ही सोचूँ
तुम्हें धन्यवाद
आपके पास

🇫🇷 Gujarati translation: આમાં હેલો, તમારી ટકાઉ દ્વારા ?

🇯🇵 Panjabi translation: ਹੇਲਾ ਕੀ? ਦਿਨ
ਜਵਾਬ: ਮੈਂ ਗੁੱਸੇ ਤੋਂ ਘੱਟ ਪਰ, ਉਹ ਅਖ਼ੀਰ ਕਿੰਨੇ ਲੱਭ ਰਹੇ?
